# 01. Extract Apache JIRA Issue Data

Pulls real, resolved issue data from the Apache Software Foundation's public JIRA
instance via its REST API. No authentication is required for public projects.

**Output:** `data/raw/apache_jira_raw.csv`

**Status: this script has already been run for real**: the output file
(`apache_jira_raw.csv`, 30,733 real issues from Camel + Hadoop) already exists in
this repository. This notebook is provided so the extraction is documented and
reproducible, not because it needs to be re-run right now. Re-running it will hit
the live Apache JIRA API again and may take several minutes.

**Known gap:** this script pulls issue/process metadata (priority, component,
comments, resolution dates) directly from the JIRA API, but **not** `num_reassignments`
or the issue `type` field (e.g., Bug vs. Improvement): both are named in the
synopsis but require a different JIRA API field (the issue changelog/history
endpoint) not yet added here. Code-level metrics (`loc`, `cyclomatic_complexity`,
`test_coverage_pct`, `prior_defect_count`) are **not** available from the JIRA API at
all: they require a separate repository-mining step against the project's real git
history (e.g., using `lizard` or `radon` on the commit that closed each issue). See
`03_data_cleaning.ipynb` for where these would be merged in once available.


In [1]:
!pip install -q requests pandas || pip install -q requests pandas --break-system-packages

In [2]:
import requests
import pandas as pd
import time

JIRA_BASE_URL = "https://issues.apache.org/jira/rest/api/2/search"

# Choose one or more large, long-running Apache projects with issue histories
# spanning both before and after 2023-01-01 (needed for the era comparison in RQ1/RQ2).
PROJECTS = ["CAMEL", "HADOOP"]

PAGE_SIZE = 100  # JIRA API max per request is typically 100


def fetch_project_issues(project_key: str) -> list:
    """Fetch all resolved issues for a given Apache project."""
    all_issues = []
    start_at = 0

    jql = f'project={project_key} AND resolution=Fixed ORDER BY resolutiondate ASC'
    fields = "created,resolutiondate,priority,components,comment,summary,status"

    while True:
        params = {
            "jql": jql,
            "startAt": start_at,
            "maxResults": PAGE_SIZE,
            "fields": fields,
        }
        response = requests.get(JIRA_BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        issues = data.get("issues", [])
        if not issues:
            break

        for issue in issues:
            f = issue["fields"]
            all_issues.append({
                "issue_id": issue["key"],
                "project_name": project_key,
                "created": f.get("created"),
                "resolution_date": f.get("resolutiondate"),
                "priority": (f.get("priority") or {}).get("name"),
                "component": ", ".join(c["name"] for c in f.get("components", [])),
                "num_comments": (f.get("comment") or {}).get("total", 0),
            })

        start_at += PAGE_SIZE
        print(f"  {project_key}: fetched {len(all_issues)} issues so far...")
        time.sleep(0.5)  # be polite to the public API

        if start_at >= data.get("total", 0):
            break

    return all_issues

## Run the extraction

**Not executed automatically in this notebook**: uncomment and run the cell below
only if you actually want to re-pull fresh data from the live JIRA API (takes a
few minutes and hits Apache's public servers). The real output already in this
repo (`data/raw/apache_jira_raw.csv`) was produced by exactly this code.

In [3]:
import os, urllib.request

GITHUB_BASE = "https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master"

def ensure_file(local_path, github_relative_path):
    """Check local disk first; if missing, pull the already-uploaded real file
    directly from the project's GitHub repo, since the real data already lives
    there. Only if BOTH of those fail does a notebook fall back to rebuilding
    from the original external source."""
    if os.path.exists(local_path):
        print(f"Found locally: {local_path}")
        return True
    os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)
    url = f"{GITHUB_BASE}/{github_relative_path}"
    try:
        print(f"Not found locally -- fetching from GitHub repo: {url}")
        req = urllib.request.Request(url, headers={"User-Agent": "qm640-capstone"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            content = resp.read()
        with open(local_path, "wb") as f:
            f.write(content)
        print(f"Downloaded {len(content)} bytes from GitHub -> {local_path}")
        return True
    except Exception as e:
        print(f"GitHub fetch failed ({e}) -- will fall back to rebuilding from source.")
        return False

ensure_file("../data/raw/apache_jira_raw.csv", "data/raw/apache_jira_raw.csv")

if os.path.exists("../data/raw/apache_jira_raw.csv"):
    df = pd.read_csv("../data/raw/apache_jira_raw.csv")
    print(f"Existing real extraction found: {len(df)} issues")
    print(df["project_name"].value_counts())
else:
    print("No existing extraction found -- uncomment the live-extraction block above to run it.")

Not found locally -- fetching from GitHub repo: https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master/data/raw/apache_jira_raw.csv
Downloaded 2894682 bytes from GitHub -> ../data/raw/apache_jira_raw.csv
Existing real extraction found: 30733 issues
project_name
CAMEL     19927
HADOOP    10806
Name: count, dtype: int64
